In [20]:
import os
import chromadb

from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_community.embeddings import HuggingFaceEmbeddings
from pathlib import Path
from dotenv import load_dotenv
from groq import Groq

In [21]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")

In [29]:
from pathlib import Path
from dotenv import load_dotenv
import os
from groq import Groq

load_dotenv(dotenv_path=r"C:\Users\nakul\OneDrive\Desktop\training\.env")

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
client = Groq(api_key=GROQ_API_KEY)

In [23]:
MODEL_NAME = "llama-3.3-70b-versatile"
TESLA_10K_COLLECTION = "tesla-10k-2019-to-2023"
CHROMA_DB_PATH = "./tesla_db"
RETRIEVER_K = 5

In [24]:

chromadb_client = chromadb.PersistentClient(path=CHROMA_DB_PATH)

vectorstore_persisted = Chroma(
    collection_name=TESLA_10K_COLLECTION,
    collection_metadata={"hnsw:space": "cosine"},
    embedding_function=embedding_model,
    client=chromadb_client,
    persist_directory=CHROMA_DB_PATH,
)

retriever = vectorstore_persisted.as_retriever(
    search_type="similarity",
    search_kwargs={"k": RETRIEVER_K},
)

print(vectorstore_persisted._collection.count())

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


3337


In [25]:
query_expansion_system_message = """
You are a financial domain expert assisting in answering questions related to 10-k reports.
Perform query expansion on the question below. If there are multiple common ways of phrasing a user question \
or common synonyms for key words in the question, make sure to return multiple versions \
of the query with the different phrasings.

If there are acronyms or words you are not familiar with, do not try to rephrase them.

Return at least 3 versions of the question as a list.
Generate only a list of questions, each question in a new line.
Do not number the list of questions or use bullet points.
Do not mention anything before or after the list.
"""

user_message_template = """
<Question>
{question}
</Question>
"""

In [26]:
qna_system_message = """
You are an assistant to a financial services firm who answers user queries on annual reports.
User input will have the context required by you to answer user queries.
This context will be delimited by: <Context> and </Context>.
The context contains references to specific portions of a document relevant to the user query.

User queries will be delimited by: <Question> and </Question>.

Please answer user queries only using the context provided in the input.
Do not mention anything about the context in your final answer. Your response should only contain the answer to the question.

If the answer is not found in the context, respond "I don't know".
"""

qna_user_message_template = """
<Context>
Here are some documents that are relevant to the question mentioned below.
{context}
</Context>

<Question>
{question}
</Question>
"""

In [27]:
def expand_query(user_query):
    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {"role": "system", "content": query_expansion_system_message},
            {"role": "user", "content": user_message_template.format(question=user_query)}
        ]
    )
    expansions = response.choices[0].message.content.strip().split("\n")
    return expansions


def retrieve_via_query_expansion(user_query):
    query_expansions_list = expand_query(user_query)

    expanded_context_list = []
    for query in query_expansions_list:
        expanded_context_list.extend([d.page_content for d in retriever.invoke(query)])

    return list(set(expanded_context_list))


def respond(user_query):
    context_list = retrieve_via_query_expansion(user_query)
    context_for_query = "\n\n".join(context_list)

    prompt = [
        {"role": "system", "content": qna_system_message},
        {"role": "user", "content": qna_user_message_template.format(
            context=context_for_query,
            question=user_query,
        )},
    ]

    try:
        response = client.chat.completions.create(
            model=MODEL_NAME,
            messages=prompt,
            temperature=0,
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        return f"Sorry, I encountered the following error:\n{e}"

In [28]:
user_query = "Any specific fines levied on the company in 2022?"
answer = respond(user_query)
print(f"Assistant: {answer}")

AuthenticationError: Error code: 401 - {'error': {'message': 'Invalid API Key', 'type': 'invalid_request_error', 'code': 'expired_api_key'}}